Keyword Extraction Using KeyBERT.

Install the relevant packages.

In [47]:
from keybert import KeyBERT

KeyBert starts by embedding a document into a vector by turning a chunk of text into a fixed sized vector which represents the semnatics of the document.
Extracts key words using simple techniques - count vectorizer, TFADF vectorizer
Embeds each word using the same model used to embed the document, leaving a list of keyword embeddings. 
Similarity measures will be calculated between document and keyword embeddings. This results in a vector, where each value is a similarity measure between the two. Sorts results in decreasing order to get the most simliar words. 

Adding diversity in the data:
Sometimes there is some redundancy in the data, and repetition. 
Max Sum Similarity
Maximal Marginal Relevance

In [48]:
model = KeyBERT(model = "distilbert-base-nli-mean-tokens")

One document PER Coordinate is needed for analysis

In [49]:
import pandas as pd

In [50]:
all_feedback = pd.read_csv('C:\\Users\\User\\Documents\\GitHub\\proutectapp\\csv-json-files\\joinedfeedback.csv')
ratings = all_feedback.loc[:, ['Latitude', 'Longitude', 'q1', 'q2', 'q3', 'q4']]
responses =  all_feedback.loc[:, ['Latitude', 'Longitude', 'responseList']]

Need to classify each coordinate to be safe/unsafe utilising user feedback. Used average of all answers to question 1 "How safe did you feel on the route?" 

In [51]:
classification_list = []
for index, row in ratings.iterrows():
    if row["q1"] > 2.5:
        classification_list.append("safe")
    else:
        classification_list.append("unsafe")

ratings["Classification"] = classification_list

ratings
ratings["Classification"].value_counts()

Classification
safe      413
unsafe    371
Name: count, dtype: int64

In [52]:
keywords_list = []

for responses in responses['responseList']:
  
    doc = f"""
    {responses}
    """

    keywords = model.extract_keywords(doc, keyphrase_ngram_range=(1,2), stop_words='english', use_mmr=True, diversity=0.9)

    keywords_list.append(keywords)

responses['Key words'] = keywords_list
    

The route was busy with the public, but there was a lot of cars which made me feel unsafe at times.; The route was busy with the public, but there was a lot of cars which made me feel unsafe at times.
There was a long, isolated road with a lack of lighting on the route.; There was a long, isolated road with a lack of lighting on the route.
There was a long, isolated road with a lack of lighting on the route.; There was a long, isolated road with a lack of lighting on the route.
The route was busy with the public as there were many shops nearby. I think the parade would have been a safer choice.; The route was well lit, busy, with lots of public spaces.; This route was very well-lit, with lots of access to shops if needed. Many people were walking around even though it's dark.; The route had lots of public spaces available so I felt at ease.; The route was busy with the public as there were many shops nearby. I think the parade would have been a safer choice.; The route was well lit, bu

Well-lit street, was quite busy with other members of public.; Well-lit street, was quite busy with other members of public.
There was a long, isolated road with a lack of lighting on the route.; There was a long, isolated road with a lack of lighting on the route.
On route there was a road which had no street lighting and felt very unsafe. It was residential, but there was no access to public spaces.; On route there was a road which had no street lighting and felt very unsafe. It was residential, but there was no access to public spaces.
The route was busy with the public, but there was a lot of cars which made me feel unsafe at times.; The route was busy with the public, but there was a lot of cars which made me feel unsafe at times.
The route was busy with the public, but there was a lot of cars which made me feel unsafe at times.; The route was isolated as it was far away from shops and any public spaces. The hospital was nearby which made me feel more safe but overall would have p

KeyboardInterrupt: 

In [ ]:
responses

,Latitude,Longitude,responseList,Key words
0,52.285500,-1.558030,"The route was busy with the public, but there ...","[(cars feel unsafe, 0.7099), (busy public, 0.5..."
1,52.286880,-1.533620,"There was a long, isolated road with a lack of...","[(long isolated road, 0.8514), (lack lighting ..."
2,52.279250,-1.544650,"There was a long, isolated road with a lack of...","[(long isolated road, 0.8514), (lack lighting ..."
3,52.290777,-1.533494,The route was busy with the public as there we...,"[(busy public shops, 0.5989), (parade, 0.3846)..."
4,52.275950,-1.516430,"Well-lit street, was quite busy with other mem...","[(public lit street, 0.7701), (street quite bu..."
...,...,...,...,...
779,52.292370,-1.548190,A really nice walk to do in the evening.; The ...,"[(park night drunk, 0.5805), (poorly lit 12am,..."
780,52.285970,-1.536280,The route felt fairly safe with some street li...,"[(park felt helpless, 0.581), (night route, 0...."
781,52.282960,-1.545510,The route felt fairly safe with some street li...,"[(park felt helpless, 0.581), (night route, 0...."
782,52.288400,-1.538270,Walking through the park made me feel uneasy a...,"[(park feel uneasy, 0.6819), (uneasy lighting,..."


In [ ]:
keywords = model.extract_keywords(doc)

In [ ]:
print(keywords)

[('drunk', 0.526), ('night', 0.3774), ('dark', 0.3303), ('park', 0.132), ('walking', 0.118)]


Removing repetition - Max_sum similarity and Maximal Marginal Relevance.

In [ ]:
model.extract_keywords(doc, keyphrase_ngram_range=(1,3), stop_words='english',
                       use_maxsum=True, nr_candidates=20, top_n=4)

[('people', 0.0513),
 ('walking park', 0.2037),
 ('drunk people', 0.5948),
 ('park night drunk', 0.7046)]

In [ ]:
model.extract_keywords(doc, keyphrase_ngram_range=(1,3), stop_words='english',
                       use_mmr=True, diversity=0.9)

[('park night drunk', 0.7046),
 ('walking park night', 0.5303),
 ('people dark', 0.4086),
 ('park', 0.132),
 ('walking', 0.118)]